# MaNGA (MMU) data breakdown

This notebook summarizes **what you have on disk**, estimates **JSON-encoded size by feature**, and outlines the **feature space** for the `MultimodalUniverse/manga` split (structure aligns with [MultimodalUniverse on GitHub](https://github.com/MultimodalUniverse/MultimodalUniverse) and the MMU paper [arXiv:2412.02527](https://arxiv.org/abs/2412.02527)).

**Is JSON bad for capacity?** For huge numeric arrays (IFU spectra, image pixels, maps), **yes—JSON is inefficient** versus binary formats:
- Numbers become decimal text (many bytes per float), plus brackets, commas, and whitespace if pretty-printed.
- The same logical payload is typically **much smaller** in **HDF5, FITS, NumPy `.npz` / Zarr / Arrow**, often by an order of magnitude or more for pure numeric data.
- Hugging Face often serves rows as Arrow/Parquet internally; **exporting each row to `.json` inflates size** versus keeping binary columns or a curated subset of fields.

With ~**2 GB per galaxy** as JSON on disk, **12k objects ≈ 24 TB** as full JSON exports—consistent with your back-of-the-envelope. Your **~3 TB** budget implies keeping only a **subset of galaxies** and/or **subset of modalities** (e.g. metadata + binned spectra, not every spaxel at full resolution), or using **streaming + on-the-fly processing** without persisting everything.

**Requirements:** `pandas`, `ijson` (streaming; avoids loading a 2 GB object twice). Install if needed:

```bash
pip install pandas ijson
```

In [ ]:
from __future__ import annotations

import json
import statistics
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

try:
    import ijson
except ImportError as e:
    raise ImportError("Install ijson: pip install ijson") from e

# Folder with sample_*.json from your download script
DATA_DIR = Path("mmu_huggingface_individual")

In [ ]:
def estimate_json_utf8_bytes(obj: Any, _depth: int = 0) -> int:
    """Rough size of obj if serialized as compact JSON (UTF-8), without building the full string.

    Fast path: homogeneous numeric lists (common for flux, wavelengths, etc.).
    """
    if _depth > 200:
        return 8  # fallback; avoid pathological recursion
    if obj is None:
        return 4  # null
    if obj is True or obj is False:
        return 4 if obj else 5
    if isinstance(obj, int):
        return len(str(obj).encode("utf-8"))
    if isinstance(obj, float):
        # json uses shortest round-trip-ish encoding; repr is a decent proxy
        return len(json.dumps(obj).encode("utf-8"))
    if isinstance(obj, str):
        return len(json.dumps(obj, ensure_ascii=False).encode("utf-8"))
    if isinstance(obj, (bytes, bytearray)):
        # JSON can't encode raw bytes; HF usually decodes; treat as utf-8 len upper bound
        return len(json.dumps(obj.decode("utf-8", errors="replace"), ensure_ascii=False).encode("utf-8"))
    if isinstance(obj, list):
        if not obj:
            return 2  # []
        n = len(obj)
        head = obj[: min(2000, n)]
        if head and all(isinstance(x, (int, float)) for x in head):
            if all(isinstance(x, (int, float)) for x in obj):
                sample = obj[: min(10_000, n)]
                per = statistics.mean(len(json.dumps(x).encode("utf-8")) for x in sample)
                inner = int(per * n)
                return 1 + inner + max(0, n - 1) + 1  # [ ... ] with commas
        # Heterogeneous or nested list: recurse (may be slow on huge lists of dicts)
        if n > 5000 and isinstance(head[0], dict):
            # assume roughly similar dict rows — sample a few
            k = min(20, n)
            sample_rows = [estimate_json_utf8_bytes(obj[i], _depth + 1) for i in range(k)]
            mean_row = statistics.mean(sample_rows)
            inner = int(mean_row * n)
            return 1 + inner + max(0, n - 1) + 1  # [ ... ] with commas
        total = 1  # '['
        for i, x in enumerate(obj):
            if i:
                total += 1  # comma
            total += estimate_json_utf8_bytes(x, _depth + 1)
        total += 1  # ']'
        return total
    if isinstance(obj, dict):
        if not obj:
            return 2
        total = 1  # '{'
        first = True
        for k, v in obj.items():
            if not first:
                total += 1
            first = False
            total += estimate_json_utf8_bytes(k, _depth + 1) + 1 + estimate_json_utf8_bytes(v, _depth + 1)
        total += 1  # '}'
        return total
    # fallbacks (e.g. nested numpy scalars if ever present)
    try:
        return len(json.dumps(obj, default=str, ensure_ascii=False).encode("utf-8"))
    except TypeError:
        return len(repr(obj).encode("utf-8"))


def human_bytes(n: float) -> str:
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024 or unit == "TiB":
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PiB"

In [ ]:
@dataclass
class MangaBreakdown:
    path: Path
    disk_bytes: int
    object_id: Any | None = None
    z: Any | None = None
    spaxel_size: Any | None = None
    spaxel_size_units: Any | None = None
    n_spaxels: int = 0
    spaxel_prototype: dict | None = None
    spaxel_keys: list | None = None
    images: list = field(default_factory=list)
    maps: list = field(default_factory=list)
    scalars_json_est: int = 0
    spaxels_block_est: int = 0
    images_block_est: int = 0
    maps_block_est: int = 0
    total_json_est: int = 0


def _one_ijson(path: Path, prefix: str):
    with path.open("rb") as f:
        return next(ijson.items(f, prefix))


def analyze_manga_json_stream(path: Path) -> MangaBreakdown:
    """Stream one MaNGA/MMU JSON row without loading the full `spaxels` list into RAM."""
    path = Path(path)
    disk = path.stat().st_size

    object_id = _one_ijson(path, "object_id")
    z = _one_ijson(path, "z")
    spaxel_size = _one_ijson(path, "spaxel_size")
    spaxel_size_units = _one_ijson(path, "spaxel_size_units")

    scalars_only = estimate_json_utf8_bytes(
        {
            "object_id": object_id,
            "z": z,
            "spaxel_size": spaxel_size,
            "spaxel_size_units": spaxel_size_units,
        }
    )
    meta_inner = scalars_only - 2

    spaxel_proto = None
    n_spx = 0
    with path.open("rb") as f:
        for spx in ijson.items(f, "spaxels.item"):
            n_spx += 1
            if spaxel_proto is None:
                spaxel_proto = spx
    if n_spx and spaxel_proto is not None:
        one_sp = estimate_json_utf8_bytes(spaxel_proto)
        spaxels_est = 1 + n_spx * one_sp + max(0, n_spx - 1) + 1
    else:
        spaxels_est = 2

    images_list: list = []
    with path.open("rb") as f:
        for img in ijson.items(f, "images.item"):
            images_list.append(img)
    images_est = estimate_json_utf8_bytes(images_list)

    maps_list: list = []
    with path.open("rb") as f:
        for m in ijson.items(f, "maps.item"):
            maps_list.append(m)
    maps_est = estimate_json_utf8_bytes(maps_list)

    pair_spax = estimate_json_utf8_bytes("spaxels") + 1 + spaxels_est
    pair_img = estimate_json_utf8_bytes("images") + 1 + images_est
    pair_map = estimate_json_utf8_bytes("maps") + 1 + maps_est
    root_est = 1 + meta_inner + 1 + pair_spax + 1 + pair_img + 1 + pair_map + 1

    content_sum = meta_inner + spaxels_est + images_est + maps_est
    scale = disk / content_sum if content_sum else 1.0

    spaxel_keys = list(spaxel_proto.keys()) if isinstance(spaxel_proto, dict) else None

    spaxel_field_raw: dict[str, float] = {}
    if isinstance(spaxel_proto, dict) and spaxel_proto:
        pair_sizes = {k: estimate_json_utf8_bytes({k: spaxel_proto[k]}) for k in spaxel_proto}
        denom = sum(pair_sizes.values()) or 1.0
        for k, ps in pair_sizes.items():
            spaxel_field_raw[k] = spaxels_est * (ps / denom)

    bd = MangaBreakdown(
        path=path,
        disk_bytes=disk,
        object_id=object_id,
        z=z,
        spaxel_size=spaxel_size,
        spaxel_size_units=spaxel_size_units,
        n_spaxels=n_spx,
        spaxel_prototype=spaxel_proto,
        spaxel_keys=spaxel_keys,
        images=images_list,
        maps=maps_list,
        scalars_json_est=int(meta_inner * scale),
        spaxels_block_est=int(spaxels_est * scale),
        images_block_est=int(images_est * scale),
        maps_block_est=int(maps_est * scale),
        total_json_est=disk,
    )
    bd.scale_to_disk = scale
    bd.root_json_est = root_est
    bd.spaxel_fields_disk = {k: int(v * scale) for k, v in spaxel_field_raw.items()}
    bd.spaxel_fields_frac_of_spaxels = {
        k: (v / spaxels_est) if spaxels_est else 0.0 for k, v in spaxel_field_raw.items()
    }
    return bd

## MaNGA row feature space (MMU)

Each Hugging Face row is a nested JSON object with **top-level keys**:

| Key | Role (high level) |
|-----|-------------------|
| `object_id` | Galaxy / MaNGA identifier |
| `z` | Redshift (scalar) |
| `spaxel_size`, `spaxel_size_units` | IFU spaxel scale |
| `spaxels` | **List of per-spaxel records** (spectra, coordinates, masks, LSF, etc.) — typically dominates disk |
| `images` | Multi-band imaging (per filter: flux, PSF, pixel scale, units) |
| `maps` | 2D map products (each with `group`, `label`, `flux`, `ivar`, `mask`, units) |

**Per-spaxel dict** (typical keys from your exploratory notebook): spectra (`flux`, `ivar`, `mask`, `lsf`, `lambda`), sky/elliptical coords (`skycoo_*`, `ellcoo_*`), `x`, `y`, `spaxel_idx`, and various `*_units` string fields.

Use the tables below to see **which branches dominate JSON size** so you can drop or downsample (e.g. omit `ivar` / LSF, bin wavelengths, store sparse spaxels only) when exporting to FITS/NPZ/Zarr.

In [ ]:
def list_sample_json_files(folder: Path | None = None) -> list[Path]:
    folder = folder or DATA_DIR
    return sorted(folder.glob("sample_*.json"))


def disk_inventory_table(paths: list[Path]) -> pd.DataFrame:
    rows = []
    for p in paths:
        st = p.stat()
        rows.append(
            {
                "file": p.name,
                "disk_MiB": st.st_size / (1024**2),
                "disk_bytes": st.st_size,
            }
        )
    df = pd.DataFrame(rows)
    if not df.empty:
        df.loc[len(df)] = [
            "TOTAL",
            df["disk_MiB"].sum(),
            int(df["disk_bytes"].sum()),
        ]
    return df


files = list_sample_json_files()
inv = disk_inventory_table(files)
inv

In [ ]:
# Set to a small number for a quick pass; None = all JSON files in DATA_DIR.
# Full pass is I/O heavy (~multi-minute per ~2 GiB file on this streaming analyzer).
MAX_FILES: int | None = None

paths = list_sample_json_files()
if MAX_FILES is not None:
    paths = paths[:MAX_FILES]

rows = []
breakdowns: list = []
for p in paths:
    bd = analyze_manga_json_stream(p)
    breakdowns.append(bd)
    rows.append(
        {
            "file": p.name,
            "object_id": bd.object_id,
            "n_spaxels": bd.n_spaxels,
            "disk_GiB": bd.disk_bytes / (1024**3),
            "metadata_bytes_est": bd.scalars_json_est,
            "spaxels_bytes_est": bd.spaxels_block_est,
            "images_bytes_est": bd.images_block_est,
            "maps_bytes_est": bd.maps_block_est,
            "scale_to_disk": round(bd.scale_to_disk, 6),
            "root_json_est": bd.root_json_est,
        }
    )

summary = pd.DataFrame(rows)
bd0 = breakdowns[0]  # detailed drill-down in following cells
summary

In [ ]:
# Uses bd0 from cell 6 (first file in `paths`). Re-run cell 6 first after changing MAX_FILES.
SAMPLE_FILE = paths[0]

top_df = pd.DataFrame(
    [
        {
            "component": "metadata (object_id, z, spaxel_size, units)",
            "approx_disk_bytes": bd0.scalars_json_est,
            "approx_disk": human_bytes(bd0.scalars_json_est),
            "pct_of_file": 100.0 * bd0.scalars_json_est / bd0.disk_bytes,
        },
        {
            "component": "spaxels[]",
            "approx_disk_bytes": bd0.spaxels_block_est,
            "approx_disk": human_bytes(bd0.spaxels_block_est),
            "pct_of_file": 100.0 * bd0.spaxels_block_est / bd0.disk_bytes,
        },
        {
            "component": "images[]",
            "approx_disk_bytes": bd0.images_block_est,
            "approx_disk": human_bytes(bd0.images_block_est),
            "pct_of_file": 100.0 * bd0.images_block_est / bd0.disk_bytes,
        },
        {
            "component": "maps[]",
            "approx_disk_bytes": bd0.maps_block_est,
            "approx_disk": human_bytes(bd0.maps_block_est),
            "pct_of_file": 100.0 * bd0.maps_block_est / bd0.disk_bytes,
        },
    ]
)
top_df

In [ ]:
# Within one spaxel dict: approximate JSON contribution per field (scaled to file; summed ≈ spaxels slice)
spax_fields = (
    pd.DataFrame(
        [
            {
                "field": k,
                "approx_disk_bytes": v,
                "pct_of_spaxels_block": 100.0 * v / bd0.spaxels_block_est,
                "frac_of_spaxels_json_est": bd0.spaxel_fields_frac_of_spaxels.get(k, 0),
            }
            for k, v in sorted(bd0.spaxel_fields_disk.items(), key=lambda kv: -kv[1])
        ]
    )
    if bd0.spaxel_fields_disk
    else pd.DataFrame()
)
spax_fields

In [ ]:
def per_item_disk_share(items: list, block_disk_bytes: int, label_key: str) -> pd.DataFrame:
    if not items:
        return pd.DataFrame()
    raw = [estimate_json_utf8_bytes(x) for x in items]
    s = sum(raw) or 1
    rows = []
    for i, x in enumerate(items):
        lbl = x.get(label_key, "") if isinstance(x, dict) else ""
        row = {
            "index": i,
            label_key: lbl,
            "approx_disk_bytes": int(block_disk_bytes * (raw[i] / s)),
            "json_est_raw": raw[i],
        }
        if isinstance(x, dict) and "group" in x:
            row["group"] = x.get("group")
        rows.append(row)
    return pd.DataFrame(rows)


per_image = per_item_disk_share(bd0.images, bd0.images_block_est, "filter")
per_map = per_item_disk_share(bd0.maps, bd0.maps_block_est, "label")
display(per_image)
display(per_map)